# 04 Feature importance per feature family

Reads the importance tables written by the modelling notebooks (one per run). For the random
forest these are the impurity based importances, for MLP and SGD the permutation importances on
the in domain test split. Importances are summed per family and normalised to a share per run.

Note: impurity importance favours features with many distinct values (here the per source
timing statistics) and is computed on the training capture, so it says what the model *uses*,
not what *transfers*. Notebook 02 (ablation) answers the second question.

In [ ]:
import sys, pathlib, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(pathlib.Path.cwd()))  # the notebooks live in code/analysis
import analysis_utils as au
importlib.reload(au)

RESULTS = pathlib.Path('results')
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

EXPS = au.available_experiments()
print('experiments with exported artifacts:', EXPS)
missing = [e for e in au.EXPERIMENTS if e not in EXPS]
if missing:
    print('NOT YET RERUN (no artifacts):', missing)

In [ ]:
rows = []
for e in EXPS:
    for f in sorted((au.ARTIFACT_DIR / e / 'importance').glob('*.csv')):
        label, model, split, frame = f.stem.split('__')
        d = pd.read_csv(f)
        d = d.assign(exp=au.exp_name(e), label=label, model=model, split=split)
        rows.append(d)
imp = pd.concat(rows, ignore_index=True)
imp['family'] = imp['Feature'].map(au.family_of)
imp.to_csv(RESULTS / 'feature_importance_all.csv', index=False)

fam = (imp.groupby(['exp', 'label', 'model', 'split', 'family'])['Importance'].sum().reset_index())
fam['share'] = fam['Importance'] / fam.groupby(['exp', 'label', 'model', 'split'])['Importance'].transform(
    lambda s: s.clip(lower=0).sum())
fam['n_features'] = fam['family'].map(lambda f: len(au.FEATURE_FAMILIES.get(f, [])))
fam['share_per_feature'] = fam['share'] / fam['n_features'].replace(0, np.nan)
fam.to_csv(RESULTS / 'feature_importance_per_family.csv', index=False)
fam[(fam['model'] == 'random_forest') & (fam['label'] == 'multi') & (fam['split'] == 'grouped')] \
    .pivot(index='family', columns='exp', values='share').round(3)

In [ ]:
for label in ['multi', 'binary']:
    h = (fam[(fam['model'] == 'random_forest') & (fam['label'] == label) & (fam['split'] == 'grouped')]
         .pivot(index='family', columns='exp', values='share'))
    if h.empty:
        continue
    h = h.loc[h.mean(axis=1).sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    im = ax.imshow(h.values, cmap='Blues', aspect='auto', vmin=0)
    ax.set_xticks(range(h.shape[1])); ax.set_xticklabels(h.columns, rotation=20, ha='right')
    ax.set_yticks(range(h.shape[0])); ax.set_yticklabels(h.index)
    for i in range(h.shape[0]):
        for j in range(h.shape[1]):
            ax.text(j, i, f'{h.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, label='share of the total importance')
    ax.set_title(f'Random forest importance per family ({label}, grouped split)')
    plt.tight_layout(); plt.savefig(FIGURES / f'fig_importance_family_{label}.pdf'); plt.show()

## Top ten features per experiment (random forest, three class, grouped split)

In [ ]:
top = (imp[(imp['model'] == 'random_forest') & (imp['label'] == 'multi') & (imp['split'] == 'grouped')]
       .sort_values('Importance', ascending=False).groupby('exp').head(10))
top[['exp', 'Feature', 'family', 'Importance']].reset_index(drop=True)

## Agreement between learners (MLP and SGD use permutation importance)

In [ ]:
fam[(fam['label'] == 'multi') & (fam['split'] == 'grouped')] \
    .pivot_table(index='family', columns=['model'], values='share', aggfunc='mean').round(3)